04-feature engineering

The goal of this notebook is to transform the raw dataset into a machine-learning-ready dataset.

The following preprocessing steps will be performed:

- Missing value handling
- Feature creation
- Categorical encoding
- Outlier treatment
- Feature scaling
- Feature selection
- Save processed dataset

3.1 Import Libraries

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import (
    StandardScaler , 
    RobustScaler , 
    MinMaxScaler , 
    LabelEncoder 
)

from sklearn.feature_selection import VarianceThreshold 
from scipy.stats import zscore 

import warnings 
warnings.filterwarnings('ignore')

3.2 Load Dataset

In [2]:
PROJECT_ROOT = Path().cwd().parent 
DATA_PATH = PROJECT_ROOT / 'data' / 'Raw'
application_train = pd.read_csv(DATA_PATH / 'application_train.csv')

In [3]:
application_train.shape

(307511, 122)

In [4]:
application_train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


3.3 Missing value handling

In [80]:
missing = (
    application_train.isnull()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending= False)
    .reset_index()
)

missing.columns = ['Feature' , 'Missing %']


def decision_strategy(pct) :
    if pct == 0 :
        return 'Keep' 
    elif pct < 5 : 
        return 'Median / Mode'
    elif pct < 40 : 
        return 'Investigate'
    else : 
        return 'Review Carefully'

missing['Decision'] = missing['Missing %'].apply(decision_strategy)
missing['Dtype'] = application_train.dtypes.values

In [81]:
missing[missing['Missing %'] >= 40]

,Feature,Missing %,Decision,Dtype
0,COMMONAREA_AVG,69.87,Review Carefully,int64
1,COMMONAREA_MEDI,69.87,Review Carefully,int64
2,COMMONAREA_MODE,69.87,Review Carefully,str
3,NONLIVINGAPARTMENTS_MEDI,69.43,Review Carefully,str
4,NONLIVINGAPARTMENTS_MODE,69.43,Review Carefully,str
5,NONLIVINGAPARTMENTS_AVG,69.43,Review Carefully,str
6,FONDKAPREMONT_MODE,68.39,Review Carefully,int64
7,LIVINGAPARTMENTS_MEDI,68.35,Review Carefully,float64
8,LIVINGAPARTMENTS_MODE,68.35,Review Carefully,float64
9,LIVINGAPARTMENTS_AVG,68.35,Review Carefully,float64


3.4 Feature Creation

In [33]:

feature_formulas = {
    "INCOME_PER_PERSON":
        lambda df: df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"],

    "CREDIT_INCOME_RATIO":
        lambda df: df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"],

    "ANNUITY_INCOME_RATIO":
        lambda df: df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"],

    "EMPLOYED_AGE_RATIO":
        lambda df: np.abs(df["DAYS_EMPLOYED"]) /
                   np.abs(df["DAYS_BIRTH"]),

    "CAR_AGE_RATIO":
        lambda df: df["OWN_CAR_AGE"] /
                   np.abs(df["DAYS_BIRTH"]),

    "EMPLOYED_PERCENT":
        lambda df: np.abs(df["DAYS_EMPLOYED"]) /
                   np.abs(df["DAYS_BIRTH"]),

    "ANNUITY_CREDIT_RATIO":
        lambda df: df["AMT_ANNUITY"] /
                   df["AMT_CREDIT"],

    "GOODS_CREDIT_RATIO":
        lambda df: df["AMT_GOODS_PRICE"] /
                   df["AMT_CREDIT"],

    "CHILDREN_RATIO":
        lambda df: df["CNT_CHILDREN"] /
                   df["CNT_FAM_MEMBERS"],

    "PHONE_CHANGE_RATIO":
        lambda df: np.abs(df["DAYS_LAST_PHONE_CHANGE"]) /
                   np.abs(df["DAYS_BIRTH"])
}



def create_features(df) :
    df = df.copy()

    for feature_name , formula in feature_formulas.items() :
        df[feature_name] = formula(df)
    return df 

In [34]:
application_train = create_features(application_train)

3.5 Encoding

In [49]:
object_columns = application_train.select_dtypes(include = 'object').columns.tolist()
print(f'Number of categorical features : {len(object_columns)}')
object_columns

Number of categorical features : 16


['NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'WEEKDAY_APPR_PROCESS_START',
 'ORGANIZATION_TYPE',
 'FONDKAPREMONT_MODE',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE',
 'EMERGENCYSTATE_MODE']

In [86]:
missing[(missing['Missing %'] >= 40) & (missing['Dtype'] == 'object')]

,Feature,Missing %,Decision,Dtype


AttributeError: 'numpy.dtypes.Int64DType' object has no attribute 'sum'